# 02 — Features and leakage (Phase 5)

Builds the leakage-safe feature and label tables from the hourly canonical
data, and demonstrates — interactively, over the real fixture data — the
leakage properties enforced by the mandatory tests in
`tests/unit/test_leakage.py`. All feature/label/dataset logic lives in
`rivercast.processing`; this notebook only calls it and inspects results
(CLAUDE.md rule 17).

> RiverCast is **educational**. Water level is relative to the local gauge
> zero — not river depth — and these forecasts must never inform real-world
> decisions.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd

from rivercast.config import load_config
from rivercast.envcheck import find_lab_root
from rivercast.processing import (
    assemble_dataset,
    build_features,
    build_labels,
    build_manifest,
    normalize_measurements,
    resample_hourly,
    run_checks,
    training_rows,
)
from rivercast.sources import FixtureGaugeSource, parse_measurements

LAB_ROOT = find_lab_root(Path.cwd())
config = load_config(LAB_ROOT / "configs" / "local.yaml")
source = FixtureGaugeSource(LAB_ROOT / "data_fixtures" / "pegelonline")

TARGET = config.station(config.target_station)
UPSTREAM = [s for s in config.stations if s.name != config.target_station]
PREFIXES = {s.uuid: s.name.lower() for s in config.stations}
print(f"target: {TARGET.name} ({TARGET.uuid})")
print(f"upstream: {[s.name for s in UPSTREAM]}")

## Build the hourly canonical table (fixture window)

Reuses the Phase 3 fixture adapter and Phase 4 normalize/resample pipeline
over the committed 2024-08 overlap window — the same window the Phase 2
spike verified has complete 15-minute coverage for all four stations.

In [ ]:
WINDOW_START = datetime(2024, 8, 1, tzinfo=timezone.utc)
WINDOW_END = datetime(2024, 8, 8, tzinfo=timezone.utc)

hourly = []
for station in config.stations:
    raw = source.fetch_raw(station.uuid, "W", WINDOW_START, WINDOW_END)
    measurements = parse_measurements(raw)
    normalized = normalize_measurements(
        measurements, station.name, station.water_body, raw.metadata.sha256, WINDOW_END
    )
    hourly += resample_hourly(
        normalized.observations,
        station.uuid,
        station.name,
        "W",
        WINDOW_START,
        WINDOW_END - timedelta(hours=1),
        tolerance_minutes=config.thresholds.data_quality.resample_tolerance_minutes,
    )

print(f"hourly rows across {len(config.stations)} stations: {len(hourly)}")
missing_hours = sum(1 for h in hourly if h.is_missing)
print(f"missing hours: {missing_hours}")

## Data-quality gate before feature generation

Rule 13 (fail closed): feature generation never runs on data that failed the
Phase 4 checks.

In [ ]:
from rivercast.processing.normalize import normalize_measurements as _renorm  # noqa: F401

canonical_all = []
for station in config.stations:
    raw = source.fetch_raw(station.uuid, "W", WINDOW_START, WINDOW_END)
    normalized = normalize_measurements(
        parse_measurements(raw), station.name, station.water_body, raw.metadata.sha256, WINDOW_END
    )
    canonical_all += normalized.observations

report = run_checks(
    canonical_all,
    hourly=hourly,
    required_station_uuids={s.uuid for s in config.stations},
    value_bounds_cm=(
        config.thresholds.data_quality.value_bounds_cm.min,
        config.thresholds.data_quality.value_bounds_cm.max,
    ),
    max_short_gap_minutes=config.thresholds.data_quality.max_short_gap_minutes,
)
print(f"quality report: passed={report.passed}, {len(report.issues)} issue(s)")
for issue in report.issues:
    print(f"  [{issue.severity}] {issue.check}: {issue.message}")
assert report.passed, "data-quality gate failed; stopping before feature generation"

## Build features and labels

In [ ]:
features = build_features(hourly, TARGET.uuid, [s.uuid for s in UPSTREAM], PREFIXES)
labels = build_labels(
    hourly,
    TARGET.uuid,
    features.index,
    config.horizons_hours,
    config.thresholds.labels.match_tolerance_minutes,
)
dataset = assemble_dataset(features, labels)
print(f"feature columns: {list(features.columns)}")
print(f"dataset shape: {dataset.shape}")
dataset.head()

## Leakage demonstration

The full, automated versions of these checks are the mandatory tests in
`tests/unit/test_leakage.py`, run on every CI build. This cell demonstrates
the same property interactively: mutating observations strictly after issue
time `t` must leave the feature row at `t` unchanged.

In [ ]:
issue_t = features.index[len(features) // 2]
row_before = features.loc[issue_t].copy()

# Corrupt every KAUB hourly reading strictly after t.
tampered_hourly = [
    h.model_copy(update={"value": -9999.0})
    if h.station_uuid == TARGET.uuid and h.hour_utc > issue_t
    else h
    for h in hourly
]
features_tampered = build_features(tampered_hourly, TARGET.uuid, [s.uuid for s in UPSTREAM], PREFIXES)
row_after = features_tampered.loc[issue_t]

unchanged = row_before.equals(row_after)
print(f"issue time: {issue_t}")
print(f"feature row unchanged after future-only mutation: {unchanged}")
assert unchanged, "LEAKAGE DETECTED: a future mutation changed a past feature row"

## Dataset manifest

`dataset_id` is content-derived: identical inputs (data + code + config)
always reproduce the same ID; any change to source data, features, or code
changes it.

In [ ]:
manifest = build_manifest(
    dataset,
    target_station_uuid=TARGET.uuid,
    input_station_uuids=[s.uuid for s in UPSTREAM],
    horizons_hours=config.horizons_hours,
    source_start_utc=WINDOW_START,
    source_end_utc=WINDOW_END,
    source_checksums=sorted({
        source.fetch_raw(s.uuid, "W", WINDOW_START, WINDOW_END).metadata.sha256
        for s in config.stations
    }),
)
print(f"dataset_id: {manifest.dataset_id}")
print(f"row_count: {manifest.row_count}")

manifest_repeat = build_manifest(
    dataset,
    target_station_uuid=TARGET.uuid,
    input_station_uuids=[s.uuid for s in UPSTREAM],
    horizons_hours=config.horizons_hours,
    source_start_utc=WINDOW_START,
    source_end_utc=WINDOW_END,
    source_checksums=manifest.source_checksums,
)
assert manifest.dataset_id == manifest_repeat.dataset_id, "dataset_id must be reproducible"
print("reproducibility check passed: identical inputs -> identical dataset_id")

## Training rows vs. full dataset

Rows near the end of the window have no future observation yet for the 6h/12h
horizon and are excluded from training — but retained in `dataset` for live
forecasting, per the Phase 5 acceptance criteria.

In [ ]:
label_cols = [f"target_level_{h}h" for h in config.horizons_hours]
trainable = training_rows(dataset, label_cols)
print(f"full dataset rows: {len(dataset)}")
print(f"trainable rows (all labels present): {len(trainable)}")
print(f"excluded (label(s) not yet available): {len(dataset) - len(trainable)}")

## Conclusion

Feature and label construction is leakage-safe (verified here and by the
mandatory automated tests), deterministic, and produces a content-traceable
dataset manifest. Next: `03_baseline_training.ipynb` (Phase 6) trains the
persistence baseline and the first candidate model against `trainable`.